# Exercise: your first GPU job on Nautilus

A complete, self-contained walkthrough for Shi Lab students. Run the cells in order,
top to bottom. You will:

0. **check your setup** — the notebook tells you exactly what is missing and how to fix it,
1. see which GPUs exist right now, and what our namespace is allowed to ask for,
2. launch a debug pod with one GPU,
3. run a sanity script on the GPU and read its energy counter,
4. convert the same work into a proper batch **Job**,
5. clean everything up.

**What you need:** `kubectl`, the `kubelogin` plugin, and a `~/.kube/config` for NRP
(README step 0). If any of those is missing or stale, **section 0 below detects it and
prints the fix** — nothing else in the notebook will run until it passes.

Every cell runs on *your laptop*: `kubectl` is what talks to the cluster. Files the
notebook generates go into `.lab-tmp/` (git-ignored), and every cluster object it creates
is named after **you**, so several students can do this exercise at the same time.

> ⏱ ~20 minutes · one small free-pool GPU while you work — **run the cleanup cells at the end.**


---
## Same environment for everyone (read once)

Version problems are the #1 time sink on a shared cluster, so this repo splits the
environment in two, deliberately:

| Where | What pins it | Why |
|---|---|---|
| **Your laptop** (control plane) | `env/requirements-notebook.txt` — `ipykernel`, nothing else | this notebook imports only the Python **standard library**, so there is no dependency to drift. `kubectl` is the only real tool it needs. |
| **The cluster** (where your code runs) | `env/Dockerfile` + `env/requirements-lab.txt`, built once and pinned **by digest** | this is where torch / transformers / CUDA versions actually break. Same image for everyone, same behaviour for everyone. |

**Before your first run, once:**

```bash
bash env/setup_local_env.sh     # creates .venv and a kernel named "Nautilus Lab (.venv)"
```

Then pick **Nautilus Lab (.venv)** in the kernel selector (top-right in VS Code).

> ⚠ **If the notebook fails before any cell runs** ("kernel failed to start", "could not
> start kernel"), you most likely have a **stale kernel** selected — one registered by an
> old project whose virtualenv has since been deleted. That is not a bug in the notebook.
> `env/setup_local_env.sh` lists every kernel on your machine and marks the dead ones.


---
## 0 · Preflight — run this first

Three cells: config, local tools, cluster access. They are the reason this notebook is
self-contained — instead of a cryptic failure you get a checklist with the fix.

> **Why this section exists:** a notebook kernel does **not** always inherit your
> terminal's `PATH`. When VS Code or Jupyter is started from the Dock/launcher,
> `!kubectl ...` can fail here (`command not found`, or
> `error: unknown command "oidc-login" for "kubectl"`) even though the exact same
> command works in your terminal. Cell 0.2 repairs `PATH` inside the kernel before
> anything else runs.


In [ ]:
# ── 0.1 · Lab config — the only cell you may want to edit ──────────────────────
import os

NAMESPACE = "shi-group-ece-advanced"      # the Shi Lab namespace on Nautilus
USER_TAG  = ""                            # leave empty → taken from your git identity
                                          # (or: export LAB_USER=<your-ucsd-username>)

# The environment that matters is the container image, not your laptop. Prefer the lab
# image pinned by DIGEST (build it with env/build_image.sh): an image tag can be
# overwritten upstream, a digest cannot — that is what makes a run reproducible.
#   export LAB_IMAGE=ghcr.io/ucsd-pier-lab/shi-lab-gpu@sha256:...
IMAGE = os.environ.get("LAB_IMAGE", "pytorch/pytorch:2.4.0-cuda12.1-cudnn9-runtime")

# Free-pool cards that are plenty for this exercise. Section 1 checks which of these
# actually exist in the cluster today; the scheduler picks whichever has a free card.
#
# Deliberately NOT here: A40, RTX-A6000, A100, H100. Those nodes advertise their own
# resource key (nvidia.com/a40, nvidia.com/rtxa6000, …) and report nvidia.com/gpu: 0,
# so a pod asking for nvidia.com/gpu below can never be placed on them — it would just
# queue longer for no reason. To use one, change the resources block to ask for that
# key instead. Check any card with:
#   kubectl get nodes -o custom-columns=\
#     'PRODUCT:.metadata.labels.nvidia\.com/gpu\.product,GPU:.status.allocatable.nvidia\.com/gpu'
GPU_PRODUCTS = [
    "NVIDIA-GeForce-RTX-2080-Ti",
    "Tesla-T4",
    "NVIDIA-L4",
    "NVIDIA-A10",
    "NVIDIA-GeForce-RTX-3090",
    "NVIDIA-L40",
    "NVIDIA-L40S",
]

CPU, MEMORY  = "2", "8Gi"                 # Nautilus policy: requests == limits
WAIT_MINUTES = 15                         # how long to wait for scheduling before giving up
print(f"config loaded — image: {IMAGE}")


In [ ]:
# ── 0.2 · Preflight A: the tools on YOUR machine ───────────────────────────────
import getpass, json, os, platform, re, shutil, subprocess, sys, time
from pathlib import Path

# A Jupyter kernel often starts with a bare PATH (no /opt/homebrew/bin, no ~/.krew/bin).
# kubectl finds its auth plugin *through PATH*, so a short PATH breaks authentication.
# Repair it here — this also fixes any `!kubectl ...` cell you run later.
_extra = ["/opt/homebrew/bin", "/opt/homebrew/sbin", "/usr/local/bin", "/usr/bin", "/bin",
          "/snap/bin", str(Path.home() / ".krew/bin"), str(Path.home() / ".local/bin"),
          str(Path.home() / "bin")]
_parts = os.environ.get("PATH", "").split(os.pathsep)
for d in _extra:
    if os.path.isdir(d) and d not in _parts:
        _parts.append(d)
os.environ["PATH"] = os.pathsep.join(_parts)

KUBECTL    = shutil.which("kubectl")
PLUGIN     = shutil.which("kubectl-oidc_login")     # what `kubelogin` installs
KUBECONFIG = Path(os.environ.get("KUBECONFIG", "").split(os.pathsep)[0]
                  or Path.home() / ".kube" / "config")

OK, NO = "OK  ", "MISSING"
print(f"[{OK if KUBECTL    else NO}] kubectl          {KUBECTL or ''}")
print(f"[{OK if PLUGIN     else NO}] kubelogin plugin {PLUGIN or '(binary name: kubectl-oidc_login)'}")
print(f"[{OK if KUBECONFIG.exists() else NO}] kube config      {KUBECONFIG}")
print(f"        python           {sys.version.split()[0]}")
print(f"        platform         {platform.system()} {platform.machine()}")

# Which kernel am I actually running in? A stale kernel (registered by a project whose
# virtualenv was deleted) is a classic "the notebook is broken" that is not the notebook.
try:
    # jupyter_client ships with ipykernel, so this works in any Jupyter kernel and does
    # not depend on the `jupyter` CLI being on PATH.
    from jupyter_client.kernelspec import KernelSpecManager
    _specs = KernelSpecManager().get_all_specs()
    _dead = [n for n, sp in _specs.items()
             if "/" in sp["spec"]["argv"][0] and not os.path.exists(sp["spec"]["argv"][0])]
    print(f"        this kernel      {sys.executable}")
    if _dead:
        print("        NOTE: dead kernels are registered on this machine (never select "
              "these): " + ", ".join(_dead))
        print("              refresh yours with: bash env/setup_local_env.sh")
except Exception:
    pass   # `jupyter` not on PATH — harmless, the notebook needs no jupyter CLI

TOOLS_OK = bool(KUBECTL and PLUGIN and KUBECONFIG.exists())
if TOOLS_OK:
    print("\nAll local tools present -> continue with 0.3")
else:
    _os = platform.system()
    print("\n" + "=" * 74)
    print("FIX THIS FIRST — run in a TERMINAL, then restart the kernel and re-run 0.2")
    print("=" * 74)
    _kubectl_cmd = {
        "Darwin":  "brew install kubectl",
        "Windows": "winget install -e --id Kubernetes.kubectl",
    }.get(_os, "curl -LO 'https://dl.k8s.io/release/$(curl -Ls https://dl.k8s.io/release/stable.txt)"
                "/bin/linux/amd64/kubectl' && sudo install kubectl /usr/local/bin/")
    _plugin_cmd = {
        "Darwin":  "brew install int128/kubelogin/kubelogin",
        "Windows": "winget install -e --id int128.kubelogin",
    }.get(_os, "krew install oidc-login   # or a release from github.com/int128/kubelogin")
    if not KUBECTL:
        print("  kubectl  :", _kubectl_cmd)
    if not PLUGIN:
        print("  kubelogin:", _plugin_cmd)
        print("             (if it IS installed, this kernel simply cannot see it: run")
        print("              `which kubectl-oidc_login` — `where` on Windows — and add that")
        print("              folder to _extra at the top of this cell, then restart the kernel)")
    if not KUBECONFIG.exists():
        print("  config  : log in at portal.nrp-nautilus.io -> Get config -> save as ~/.kube/config")
        print(f"            then: kubectl config set-context nautilus --namespace={NAMESPACE}")


In [ ]:
# ── 0.3 · Preflight B: can you reach the cluster? ───────────────────────────────
def kube(*args, timeout=180, show=True):
    """Run kubectl against the lab namespace. Returns (returncode, stdout, stderr)."""
    if not KUBECTL:
        raise RuntimeError("kubectl not found — fix cell 0.2, then restart the kernel")
    cmd = [KUBECTL, *[str(a) for a in args], "-n", NAMESPACE]
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        rc, out, err = p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired:
        rc, out, err = 124, "", f"kubectl timed out after {timeout}s"
    if show:
        if out.strip():
            print(out.rstrip())
        # kubectl writes "No resources found in <ns>." to stderr and still exits 0.
        # Only printing stderr when rc != 0 turns that into a silent blank line, which
        # is exactly what you do NOT want from a cell whose job is to show you state.
        if err.strip() and (rc or not out.strip()):
            print(("!! " if rc else "   ") + err.strip()[:1500])
    return rc, out, err

# Symptom -> what it actually means. This table is the point of the preflight.
_DIAG = [
    (r'unknown command "oidc-login"',
     "kubectl cannot find the kubelogin plugin.\n"
     "     -> not installed: see cell 0.2\n"
     "     -> installed but invisible to this kernel: restart the kernel, re-run 0.2"),
    (r"Unauthorized|must be logged in|invalid_grant|refresh token|expired",
     "your login is stale.\n"
     "     -> in a terminal: kubectl oidc-login clean\n"
     "     -> then re-run this cell; a browser window opens for the UCSD login"),
    (r"forbidden|cannot list resource|is not allowed",
     f"you are logged in but not a member of `{NAMESPACE}`.\n"
     "     -> email Prof. Shi (yyshi@ucsd.edu) to be added to the lab workspace"),
    (r"timed out|i/o timeout|no such host|connection refused|TLS handshake",
     "no answer from the cluster. Two very different causes:\n"
     "     -> a browser login window is waiting: finish it and re-run this cell\n"
     "     -> or the API is unreachable: check internet/VPN and retry.\n"
     "     First login ever? Do it once from a terminal (`kubectl get pods`), where the\n"
     "     browser flow and any keychain prompt are easier to complete, then come back."),
    (r"no configuration has been provided|context .* does not exist|no such file",
     "no usable kube config -> download it from portal.nrp-nautilus.io into ~/.kube/config"),
]

CLUSTER_OK = False
if not TOOLS_OK:
    print("Skipped: fix cell 0.2 first.")
else:
    ctx = subprocess.run([KUBECTL, "config", "current-context"],
                         capture_output=True, text=True).stdout.strip()
    print(f"context   : {ctx or '(none)'}")
    print(f"namespace : {NAMESPACE}  (pinned explicitly on every command below)")
    print("checking access… if your login is not cached yet, a BROWSER WINDOW opens for the\n"
          "            UCSD login — complete it and this cell continues (up to 5 min).")
    # First login of the day goes through the browser, so this needs a generous timeout:
    # 120s would time out mid-login and look like a network failure.
    rc, out, err = kube("get", "pods", timeout=300, show=False)
    if rc == 0:
        CLUSTER_OK = True
        print("access    : OK\n")
        print("what is running in the namespace right now (shared with the whole lab):")
        print(out.rstrip() if out.strip() else "  (nothing — an empty list is fine)")
    else:
        blob = (err + out).lower()
        print("access    : FAILED\n")
        print((err or out).strip()[:800] + "\n")
        hit = next((msg for pat, msg in _DIAG if re.search(pat, blob, re.I)), None)
        print("Diagnosis: " + (hit or "unrecognised error — paste it in a lab issue "
                                      "(github.com/d2rojas/nautilus-tutorial/issues)"))

# Names + scratch dir. The namespace is shared, so everything you create is tagged with
# YOUR identity: the notebook never touches anyone else's objects, and the lab rule
# "the job name identifies you + the experiment" holds automatically.
#
# Your laptop's login name is often NOT your lab identity (a shared or inherited machine
# account), so the tag comes from your git identity first — the same identity the lab
# already requires you to commit runs with:
def _identity():
    if USER_TAG:
        return USER_TAG
    if os.environ.get("LAB_USER"):
        return os.environ["LAB_USER"]
    for args in (["config", "user.email"], ["config", "user.name"]):
        try:
            v = subprocess.run(["git", *args], capture_output=True, text=True,
                               timeout=15).stdout.strip()
        except Exception:
            v = ""
        if v:
            return v.split("@")[0]
    return getpass.getuser()          # last resort: the machine account

TAG      = (re.sub(r"[^a-z0-9]+", "-", _identity().lower()).strip("-") or "student")[:20].strip("-")
POD_NAME = f"{TAG}-tutorial-pod"
JOB_NAME = f"{TAG}-tutorial-job"
TMPDIR   = Path(".lab-tmp"); TMPDIR.mkdir(exist_ok=True)
print(f"\nyou         : {TAG}   (override: USER_TAG in cell 0.1, or export LAB_USER=...)")
print(f"your objects: pod/{POD_NAME}   job/{JOB_NAME}   (generated files in {TMPDIR}/)")
assert CLUSTER_OK, "Preflight failed — read the diagnosis above before running any other cell."


---
## 1 · What GPUs exist, and what may we ask for?

Two different questions, and both bite students:

* **Which products exist** — nodes are labeled with `nvidia.com/gpu.product`; you select a
  GPU by *node affinity* on that label. This is the live truth (also on the
  [NRP resources page](https://portal.nrp-nautilus.io/resources)).
* **What our namespace is allowed to request** — premium cards (A100/H100/GH200/H200) need
  an explicit quota grant. Without one the quota is `0/0`: a pod asking for them stays
  `Pending` forever. The second cell shows our real limits.


In [ ]:
# ── 1.1 · GPU products in the cluster right now ────────────────────────────────
import collections
rc, out, _ = kube("get", "nodes", "-o",
                  "custom-columns=GPU:.metadata.labels.nvidia\\.com/gpu\\.product",
                  "--no-headers", timeout=120, show=False)
counts = collections.Counter(l.strip() for l in out.splitlines()
                             if l.strip() and l.strip() != "<none>")
print(f"{'GPU product':<34} {'nodes':>5}   in our request list?")
for prod, n in counts.most_common(18):
    print(f"{prod:<34} {n:>5}   {'yes' if prod in GPU_PRODUCTS else ''}")

missing = [g for g in GPU_PRODUCTS if g not in counts]
if missing:
    print("\nNote: these entries of GPU_PRODUCTS do not exist in the cluster today "
          "(harmless — extra names in an affinity list are simply never matched):")
    for g in missing:
        print("  -", g)


In [ ]:
# ── 1.2 · Our namespace quota (what is NOT available to us) ────────────────────
rc, out, _ = kube("get", "resourcequota", show=False)
print(out.rstrip() if out.strip() else "(no quota objects — free pool only)")
print("\nRead it as request/limit: `0/0` means this namespace may not request that "
      "resource at all.\nThe free pool (consumer + workstation cards, up to 48 GB) is "
      "what schedules for us — which is exactly what GPU_PRODUCTS asks for.")


---
## 2 · Write the pod spec — from this notebook

The cell below **generates** the YAML (that is why the pod name has your tag in it) and
prints it. Read it before running: it asks for **one** free-pool GPU, sets
`requests == limits` as Nautilus policy requires, installs the NVML binding, prints
`READY_FOR_CODE`, then sleeps so we can drive it by hand.

> ⚠ **Lab rule:** a sleeping pod is for *debug sessions only* — never to reserve a GPU and
> "train later". Real runs go in a Job (section 4). Delete this pod the same day.


In [ ]:
# ── 2.1 · Generate the pod YAML ────────────────────────────────────────────────
gpu_values = "\n".join(f"                  - {g}" for g in GPU_PRODUCTS)
POD_YAML = TMPDIR / f"{POD_NAME}.yaml"
POD_YAML.write_text(f"""apiVersion: v1
kind: Pod
metadata:
  name: {POD_NAME}
  labels:
    owner: {TAG}
    purpose: tutorial
spec:
  restartPolicy: Never
  containers:
    - name: main
      image: {IMAGE}
      command: ["/bin/bash", "-lc"]
      args:
        - |
          set -euo pipefail
          echo "=== node: $(hostname) ==="
          nvidia-smi -L
          pip install -q nvidia-ml-py || echo "WARN: pip install failed, NVML may be missing"
          mkdir -p /workspace /output
          echo READY_FOR_CODE
          sleep infinity
      resources:
        limits:   {{nvidia.com/gpu: "1", cpu: "{CPU}", memory: {MEMORY}}}
        requests: {{nvidia.com/gpu: "1", cpu: "{CPU}", memory: {MEMORY}}}
      volumeMounts:
        - {{name: workspace, mountPath: /workspace}}
        - {{name: output, mountPath: /output}}
  affinity:
    nodeAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
        nodeSelectorTerms:
          - matchExpressions:
              - key: nvidia.com/gpu.product
                operator: In
                values:
{gpu_values}
  volumes:
    - {{name: workspace, emptyDir: {{}}}}
    - {{name: output, emptyDir: {{}}}}
""")
print(f"wrote {POD_YAML}\n")
print(POD_YAML.read_text())


---
## 3 · Launch it, wait, and run something on the GPU

`apply` submits the pod. The wait helper polls until the container prints
`READY_FOR_CODE`, and — this is the part that saves you — **stops early and shows the
scheduler's events** when the pod is stuck or broken, instead of printing `Pending` for
ten minutes. Scheduling normally takes 1–5 minutes.


In [ ]:
# ── 3.1 · Wait helper: knows the difference between "slow" and "stuck" ─────────
_BLOCKED = {"ImagePullBackOff", "ErrImagePull", "InvalidImageName",
            "CreateContainerConfigError", "CreateContainerError", "CrashLoopBackOff"}

def describe_tail(kind, name, lines=22):
    rc, out, _ = kube("describe", kind, name, show=False)
    print(f"\n--- kubectl describe {kind} {name} (tail) — read the Events ---")
    print("\n".join(out.splitlines()[-lines:]))

def wait_for_pod(pod, marker="READY_FOR_CODE", minutes=WAIT_MINUTES, every=10):
    waited, deadline, last_note = 0, minutes * 60, ""
    while waited <= deadline:
        rc, js, err = kube("get", "pod", pod, "-o", "json", show=False)
        if rc:
            print(f"[{waited:4d}s] pod not visible yet ({err.strip()[:80]})")
        else:
            st = json.loads(js).get("status", {})
            phase = st.get("phase", "?")
            cs = (st.get("containerStatuses") or [{}])[0]
            waiting = ((cs.get("state") or {}).get("waiting") or {})
            reason = waiting.get("reason", "") or st.get("reason", "")
            if phase == "Running":
                _, logs, _ = kube("logs", pod, show=False)
                if marker in logs:
                    print(logs.rstrip())
                    node = json.loads(js)["spec"].get("nodeName", "?")
                    print(f"\n--- pod is ready on node {node} ---")
                    return True
            if phase in ("Failed", "Succeeded") or reason in _BLOCKED:
                print(f"\nStopped waiting: phase={phase} reason={reason or '-'}")
                _, logs, _ = kube("logs", pod, show=False)
                if logs.strip():
                    print("\n--- container logs ---\n" + logs.rstrip())
                describe_tail("pod", pod)
                return False
            note = ""
            if phase == "Pending":
                sched = next((c for c in (st.get("conditions") or [])
                              if c.get("reason") == "Unschedulable"), None)
                if sched:
                    msg = " ".join(sched.get("message", "").split())[:150]
                    if msg != last_note:
                        note, last_note = f"\n        scheduler: {msg}…", msg
            print(f"[{waited:4d}s] phase={phase} {reason}".rstrip() + note)
        time.sleep(every); waited += every
    print(f"\nStill not ready after {minutes} min. On the free pool this usually means "
          "every card matching GPU_PRODUCTS is busy right now — widen the list (section 1) "
          "or retry later. If the events below say Unschedulable / Insufficient "
          "nvidia.com/gpu, your YAML is fine: the cluster is full.")
    describe_tail("pod", pod)
    return False

print("helpers defined")


In [ ]:
# ── 3.2 · Submit the pod and wait ──────────────────────────────────────────────
kube("delete", "pod", POD_NAME, "--ignore-not-found")     # re-runnable: clear a stale one
kube("apply", "-f", str(POD_YAML))
pod_ready = wait_for_pod(POD_NAME)


### 3.3 · A sanity script on the GPU

`hello_gpu.py` reports which card you got, whether you have it **exclusively**, and whether
the **NVML energy counter** works — the counter behind the lab's energy measurements. The
notebook writes it, copies it into the pod, and runs it: submit → wait → copy code in →
execute → read results is the whole debug-pod cycle.


In [ ]:
# ── 3.3 · Write, copy, run ─────────────────────────────────────────────────────
HELLO = TMPDIR / "hello_gpu.py"
HELLO.write_text('''import pynvml

pynvml.nvmlInit()
h = pynvml.nvmlDeviceGetHandleByIndex(0)

name = pynvml.nvmlDeviceGetName(h)
name = name.decode() if isinstance(name, bytes) else name   # bytes on older nvidia-ml-py
mem = pynvml.nvmlDeviceGetMemoryInfo(h)
procs = pynvml.nvmlDeviceGetComputeRunningProcesses(h)

print(f"GPU        : {name}")
print(f"VRAM       : {mem.total / 2**30:.1f} GiB total, {mem.used / 2**30:.1f} used")
print(f"power now  : {pynvml.nvmlDeviceGetPowerUsage(h) / 1000:.1f} W")
print(f"exclusive  : {len(procs) == 0}  ({len(procs)} other compute processes)")

try:
    e = pynvml.nvmlDeviceGetTotalEnergyConsumption(h)
    print(f"energy ctr : OK ({e/1000:.0f} J since driver load)")
except pynvml.NVMLError as ex:
    print(f"energy ctr : NOT available ({ex}) - measure energy as the integral of power")

print("HELLO_GPU_DONE")
''')

if not pod_ready:
    print("Pod is not ready - skipping. Read the events printed by 3.2 first.")
else:
    kube("cp", str(HELLO), f"{POD_NAME}:/workspace/hello_gpu.py")
    kube("exec", POD_NAME, "--", "python", "/workspace/hello_gpu.py")


**Checkpoint.** For anything that runs longer than a minute, launch it *detached* so a
dropped `exec` (laptop sleeps, network resets) cannot kill it — and poll a log file:

```bash
kubectl -n shi-group-ece-advanced exec <pod> -- bash -c \
  'nohup python /workspace/long.py > /output/run.log 2>&1 & echo started'
kubectl -n shi-group-ece-advanced exec <pod> -- tail -5 /output/run.log
```

A `kubectl exec` that hangs does **not** mean your job died — check `kubectl get pod` and
the log file.


---
## 4 · Delete the pod — then do it the *right* way: a Job

A **batch Job** is the lab default for real runs: the command lives *in the YAML*, it runs
to completion, retries once on failure, and is the record of exactly what ran. No sleeping
pod, no manual steps.


In [ ]:
# ── 4.1 · Delete the debug pod (lab rule: same day, always) ────────────────────
_ = kube("delete", "pod", POD_NAME, "--ignore-not-found")


In [ ]:
# ── 4.2 · Generate and submit the Job ──────────────────────────────────────────
job_values = "[" + ", ".join(GPU_PRODUCTS) + "]"
JOB_YAML = TMPDIR / f"{JOB_NAME}.yaml"
JOB_YAML.write_text(f"""apiVersion: batch/v1
kind: Job
metadata:
  name: {JOB_NAME}
  labels: {{owner: {TAG}, purpose: tutorial}}
spec:
  backoffLimit: 1
  template:
    metadata:
      labels: {{owner: {TAG}, purpose: tutorial}}
    spec:
      restartPolicy: Never
      affinity:
        nodeAffinity:
          requiredDuringSchedulingIgnoredDuringExecution:
            nodeSelectorTerms:
              - matchExpressions:
                  - key: nvidia.com/gpu.product
                    operator: In
                    values: {job_values}
      containers:
        - name: main
          image: {IMAGE}
          env:
            - {{name: IMAGE_REF, value: "{IMAGE}"}}
          command: ["/bin/bash", "-lc"]
          args:
            - |
              set -euo pipefail
              nvidia-smi -L
              # stamp the environment into the log: the run's reproducibility record
              echo "image      : $IMAGE_REF"
              python -c "import sys; print('python     :', sys.version.split()[0])"
              pip list --format=freeze 2>/dev/null | grep -E "^(torch|transformers|numpy|nvidia-ml-py)==" || true
              pip install -q nvidia-ml-py
              python - <<'PY'
              import pynvml
              pynvml.nvmlInit()
              h = pynvml.nvmlDeviceGetHandleByIndex(0)
              n = pynvml.nvmlDeviceGetName(h)
              print('GPU:', n.decode() if isinstance(n, bytes) else n)
              print('JOB_DONE')
              PY
          resources:
            limits:   {{nvidia.com/gpu: "1", cpu: "{CPU}", memory: {MEMORY}}}
            requests: {{nvidia.com/gpu: "1", cpu: "{CPU}", memory: {MEMORY}}}
""")
print(JOB_YAML.read_text())
kube("delete", "job", JOB_NAME, "--ignore-not-found")
_ = kube("apply", "-f", str(JOB_YAML))


In [ ]:
# ── 4.3 · Wait for the Job, with the same stuck-vs-slow logic ──────────────────
def job_pods(job):
    """Names of the pods this Job currently owns (it may briefly own none)."""
    _, out, _ = kube("get", "pods", "-l", f"job-name={job}",
                     "-o", "jsonpath={.items[*].metadata.name}", show=False)
    return out.split()

def wait_for_job(job, minutes=WAIT_MINUTES, every=10):
    """Wait for a Job — and say WHY it is waiting.

    A Job's own status only tells you how many pods are active, which is never the
    reason you are stuck. The reason lives on the *pod*, so look there too, exactly
    like wait_for_pod does in section 3.
    """
    waited, deadline, last_note = 0, minutes * 60, ""
    while waited <= deadline:
        rc, js, err = kube("get", "job", job, "-o", "json", show=False)
        if rc:
            print(f"[{waited:4d}s] job not visible yet ({err.strip()[:80]})")
        else:
            st = json.loads(js).get("status", {})
            # Read the Job's CONDITIONS, not its counters. status.failed is the number of
            # pods that have died so far — it is already 1 while a backoffLimit: 1 Job is
            # happily retrying, so branching on it declares defeat far too early.
            cond = {c.get("type"): c.get("status") for c in (st.get("conditions") or [])}
            if cond.get("Complete") == "True" or st.get("succeeded"):
                print("--- job succeeded; last 40 log lines: ---")
                kube("logs", f"job/{job}", "--tail=40")
                return True
            if cond.get("Failed") == "True":
                print("--- job FAILED (gave up after its retries); logs + events: ---")
                kube("logs", f"job/{job}", "--tail=40")
                for p in job_pods(job):
                    describe_tail("pod", p)
                return False

            note, pods = "no pod yet", job_pods(job)
            if pods:
                rc2, pjs, _ = kube("get", "pod", pods[0], "-o", "json", show=False)
                if rc2 == 0:
                    pst = json.loads(pjs).get("status", {})
                    phase = pst.get("phase", "?")
                    cs = (pst.get("containerStatuses") or [{}])[0]
                    reason = ((cs.get("state") or {}).get("waiting") or {}).get("reason", "")
                    note = f"{pods[0]} {phase} {reason}".rstrip()
                    if reason in _BLOCKED:
                        print(f"\nStopped waiting: the Job's pod is {reason}.")
                        kube("logs", f"job/{job}")
                        describe_tail("pod", pods[0])
                        return False
                    sched = next((c for c in (pst.get("conditions") or [])
                                  if c.get("reason") == "Unschedulable"), None)
                    if sched:
                        msg = " ".join(sched.get("message", "").split())[:150]
                        if msg != last_note:
                            note += f"\n        scheduler: {msg}…"
                            last_note = msg
            retried = st.get("failed", 0)
            tail = f" · {retried} attempt(s) failed so far, still retrying" if retried else ""
            print(f"[{waited:4d}s] active={st.get('active', 0)} · {note}{tail}")
        time.sleep(every); waited += every

    print(f"\nStill not finished after {minutes} min. On the free pool this usually means "
          "every card matching GPU_PRODUCTS is busy right now — widen the list (section 1) "
          "or retry later. If the events below say Unschedulable / Insufficient "
          "nvidia.com/gpu, your YAML is fine: the cluster is full.")
    pods = job_pods(job)
    if pods:
        describe_tail("pod", pods[0])
    else:
        print("The Job owns no pod at this instant — the controller is between attempts.")
        kube("get", "pods", "-l", f"job-name={job}")
    return False

job_ok = wait_for_job(JOB_NAME)


---
## 5 · Final cleanup + your checklist

Leave the namespace as you found it. This cell is safe to re-run at any time, and only ever
touches objects tagged with **your** name.


In [ ]:
# ── 5.1 · Delete everything this notebook created ──────────────────────────────
_ = kube("delete", "job", JOB_NAME, "--ignore-not-found")
_ = kube("delete", "pod", POD_NAME, "--ignore-not-found")
print("\nyour remaining objects in the namespace (should be empty):")
_ = kube("get", "pods,jobs", "-l", f"owner={TAG}")
print("\neverything in the namespace (the whole lab shares it):")
_ = kube("get", "pods,jobs")


**You now know the full loop.** Before your first real run, go back to the README:

- Step 1 — the lab workflow (local debug → smoke test → full run → cleanup)
- Step 6 — put checkpoints on the **PVC** (this exercise used `emptyDir`: everything died
  with the pod, on purpose)
- Step 7 — secrets for HF/W&B tokens
- Step 9 — the errors table, *before* you hit them
- Step 10 — the lab rules checklist (tracker, GitHub commit, smoke test, cleanup)

Hit an error the preflight did not explain?
[Open an issue](https://github.com/d2rojas/nautilus-tutorial/issues/new/choose) with the
symptom and the fix — that is how the diagnosis table in cell 0.3 was built.
